# LDA

### Construction of vocubulary (tokens)

Same approach as in graph experiment and frequential analysis

In [1]:
from utils.ml import data_dir, spark, tokenize

In [2]:
df = spark.read.csv(data_dir, header=True, inferSchema=True, sep = ";")
df, tokens = tokenize(df)

TOKENIZED DF:
+----+-----+---+-----+-------+----------+--------------------+
|year|month|day|order|country|session ID|            sequence|
+----+-----+---+-----+-------+----------+--------------------+
|2008|    4|  1|    9|     29|         1|[120, 168, 197, 8...|
|2008|    4|  1|   10|     29|         2|[143, 55, 183, 27...|
+----+-----+---+-----+-------+----------+--------------------+
only showing top 2 rows

TOKEN INFO: 
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|                     1|                     A5|     3|       2|                1|   43|      2|   1|  0|
|                     3|                    C30|     2|       4|                2|   28|      2|   2|  1|
+----------------------+----------------

In [3]:
from pyspark.sql import functions as F
from pyspark.ml.feature import HashingTF
from pyspark.ml.clustering import LDA
from pyspark.ml import Pipeline
num_features = tokens.count()
count_vectorizer = HashingTF(numFeatures=num_features, inputCol="sequence", outputCol="tf")
tf = count_vectorizer.transform(df)
tf.show(5)

+----+-----+---+-----+-------+----------+--------------------+--------------------+
|year|month|day|order|country|session ID|            sequence|                  tf|
+----+-----+---+-----+-------+----------+--------------------+--------------------+
|2008|    4|  1|    9|     29|         1|[120, 168, 197, 8...|(218,[4,37,76,82,...|
|2008|    4|  1|   10|     29|         2|[143, 55, 183, 27...|(218,[18,30,34,78...|
|2008|    4|  1|    6|     21|         3|[87, 74, 11, 188,...|(218,[4,5,53,103,...|
|2008|    4|  1|    4|     21|         4|    [90, 4, 74, 166]|(218,[5,126,154,2...|
|2008|    4|  1|    1|      9|         5|                [62]|    (218,[86],[1.0])|
+----+-----+---+-----+-------+----------+--------------------+--------------------+
only showing top 5 rows



In [4]:
scores = []
best_i = 0
first = True
for i, k in enumerate(range(2, 10)):
    lda = LDA(featuresCol="tf", maxIter=30, k = k, seed = 42)
    model = lda.fit(tf)
    perplexity = model.logPerplexity(tf)
    ll = model.logLikelihood(tf)
    scores.append({"k": k, "lp": perplexity, "ll": ll})
    print("P:", perplexity, "LL:", ll)
    if first:
        first = False
        continue
    if scores[best_i]["ll"] < ll and scores[best_i]["lp"] > perplexity:
        print("Best K:", k)
        best_i = i

P: 4.558354871561149 LL: -754289.2140167096
P: 4.555543794072213 LL: -753824.0537803054
Best K: 3
P: 4.579839580129342 LL: -757844.3746823226
P: 4.5966131326631094 LL: -760619.9615142954
P: 4.589686647288345 LL: -759473.8082733916
P: 4.631552127471577 LL: -766401.4567412318
P: 4.666390062683234 LL: -772166.2292324455
P: 4.652787259392022 LL: -769915.3189606353


In [5]:
pipeline = Pipeline(stages = [count_vectorizer, LDA(featuresCol="tf", maxIter=30, k = scores[best_i]["k"], seed = 42)])
fitted = pipeline.fit(df)
fitted.write().overwrite().save("LDA_pipeline")